# RDD Transformations and Actions: From Map to ReduceByKey

**Module**: 02_rdd  
**Notebook**: 02_rdd_transformations_and_actions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete practical catalog of RDD transformations (map, flatMap, filter, distinct, union, subtract) and actions (collect, count, take, reduce, aggregate, saveAsTextFile).
- **Why does this topic exist?**: Understanding low-level functional primitives clarifies how data moves across partitions and why map-side combining matters.
- **What problem does it solve?**: Using inefficient transformations like groupByKey causes cluster memory exhaustion.
- **Where is it used in real-world Data Engineering?**: Legacy pipeline maintenance, custom partitioning, and low-level data extraction.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | RDD fundamentals |
| **Required depth** | Working knowledge: can choose transformations and avoid unnecessary shuffles |
| **Data Engineer** | Write RDD pipelines and explain lazy evaluation, narrow/wide dependencies, and combiners. |
| **Senior Data Engineer** | Choose `reduceByKey`/`aggregateByKey` over `groupByKey` and reason about shuffle cost. |
| **Spark specialist** | Explain map-side combining, serialization, and task retries. |
| **Next topic** | `02_rdd/03_rdd_partitions.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The difference between `map` and `groupByKey` is not syntax; it is whether data must cross the network.
A pipeline that looks concise can create a massive shuffle, and a `groupByKey` can materialize every value for a
key in memory. Transformations and actions are the vocabulary for reasoning about that cost.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Narrow transformation** | No data movement between partitions | Each parent partition feeds at most one child partition |
| **Wide transformation** | Requires a shuffle | Multiple child partitions depend on the same parent partition |
| **Lazy evaluation** | Nothing runs until an action | The plan is recorded as a DAG |
| **Combiner** | Pre-aggregation before the shuffle | `reduceByKey` combines locally on the map side |
| **Action** | Triggers execution | `count`, `collect`, `take`, `reduce`, `saveAsTextFile` |
| **Lineage** | How to rebuild a lost partition | Parent RDD dependencies |

### How it works, step by step

1. Transformations append nodes to the RDD graph.
2. An action submits a job to the DAGScheduler.
3. Narrow transformations are pipelined within a stage.
4. Wide transformations write shuffle map outputs and create a new stage.
5. The reduce side fetches blocks and completes the aggregation.
6. Failed tasks are retried; lost shuffle outputs may require recomputing a map stage.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Transformations vs Actions Categorization

```
+-------------------------------------------------------------------------------+
| RDD OPERATIONS                                                                |
+---------------------------------------+---------------------------------------+
| TRANSFORMATIONS (Lazy, Return RDD)    | ACTIONS (Eager, Return Value / Write) |
+---------------------------------------+---------------------------------------+
| Narrow:                               | Collection:                           |
|  - map(func)                          |  - collect()                          |
|  - flatMap(func)                      |  - take(n), first(), top(n)           |
|  - filter(func)                       | Aggregation:                          |
|  - union(otherRdd)                    |  - count()                            |
| Wide (Shuffle):                       |  - reduce(func)                       |
|  - distinct()                         |  - aggregate(zero, seqOp, combOp)     |
|  - reduceByKey(func)                  | Output:                               |
|  - groupByKey()                       |  - saveAsTextFile(path)               |
|  - sortByKey()                        |  - foreach(func)                      |
+---------------------------------------+---------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Transformations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Key Transformations in Action

Let us test standard transformations on customer data from `/opt/spark-data/retail/customers.csv`.

In [ ]:
sc = spark.sparkContext
raw_customers = sc.textFile("/opt/spark-data/retail/customers.csv")
header = raw_customers.first()

# Filter out header and parse CSV lines into tuples: (customer_id, country, region, segment, loyalty_tier)
customers_rdd = (
    raw_customers
    .filter(lambda row: row != header)
    .map(lambda row: row.split(","))
    .map(lambda c: (int(c[0]), c[4], c[5], c[6], c[8]))
)

print(f"Total parsed customers: {customers_rdd.count()}")
print("Sample parsed customer records:")
for cust in customers_rdd.take(3):
    print(" ", cust)

## 4. PairRDD Operations: `reduceByKey` vs `aggregateByKey`

PairRDDs enable distributed key-value computations.

In [ ]:
# Key by loyalty_tier -> count customers per tier using reduceByKey
tier_counts = (
    customers_rdd
    .map(lambda c: (c[4], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

print("Customers per loyalty tier (reduceByKey):")
for tier, count in sorted(tier_counts, key=lambda x: x[1], reverse=True):
    print(f"  {tier:15}: {count}")

---

## Experiment: reduceByKey (Map-side Combine) vs groupByKey Performance

- **1. What are we testing?**: Compare execution and network shuffle volume between reduceByKey and groupByKey on PairRDDs.
- **2. What do we expect?**: reduceByKey combines values locally before network shuffle; groupByKey transfers all values across network, generating far more shuffle data.

In [ ]:
# Generate sample key-value dataset with repetitive keys
pairs = sc.parallelize([(f"Key_{i % 5}", i) for i in range(50000)], 4)

# Test reduceByKey
t0 = time.time()
r_res = pairs.reduceByKey(lambda a, b: a + b).collect()
t_reduce = time.time() - t0

# Test groupByKey + sum
t0 = time.time()
g_res = pairs.groupByKey().mapValues(sum).collect()
t_group = time.time() - t0

print(f"reduceByKey duration: {t_reduce:.4f}s")
print(f"groupByKey duration:  {t_group:.4f}s")
print("Results match:", sorted(r_res) == sorted(g_res))

**3. What happened?**: reduceByKey executed significantly faster and transferred orders of magnitude less data over network.  
**4. Why did it happen?**: reduceByKey applies the combiner function locally in the mapper memory buffer prior to shuffle write.  
**5. Practical Engineering Lesson**: Never use groupByKey() to compute aggregations; always use reduceByKey() or aggregateByKey().

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Clickstream sessionization

- **Industry**: E-commerce
- **Scenario**: A retailer groups page-view events by session to compute pages per session and conversion.
- **Problem**: `groupByKey` pulls all events for a session into one task, causing memory pressure for heavy sessions.
- **Why this concept is relevant**: `reduceByKey`/`aggregateByKey` can combine locally before the shuffle and bound the data moved per key.
- **Implementation idea**: Parse events, map to `(session_id, metrics)`, aggregate with `aggregateByKey`, and write session summaries.
- **Expected result**: Session metrics with lower shuffle volume and fewer OOMs.
- **Alternative**: A streaming sessionization job in Flink or Kafka Streams.
- **Trade-offs**: RDD aggregation is transparent for batch; streaming engines handle unbounded sessions more naturally.

### Use Case 2: Legacy word count at scale

- **Industry**: Media / publishing
- **Scenario**: A publisher counts term frequency across millions of documents for search indexing.
- **Problem**: A naive `groupByKey` implementation creates hot keys for common words and skews the reduce stage.
- **Why this concept is relevant**: `reduceByKey` performs map-side combining, reducing network traffic for frequent terms.
- **Implementation idea**: Tokenize with `flatMap`, map to `(term, 1)`, and `reduceByKey`; optionally filter stop words before the shuffle.
- **Expected result**: A term-frequency table with a smaller shuffle and fewer stragglers.
- **Alternative**: A search engine's native indexer or MapReduce.
- **Trade-offs**: RDDs are simple for batch text; a dedicated indexer is better for production search.

---

## Business Value

- Lower shuffle volume can reduce runtime and network cost.
- Fewer OOMs improve pipeline reliability.
- Clear transformation choices make code reviews easier.
- Better understanding of lazy evaluation reduces unnecessary recomputation.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Prefer `reduceByKey`/`aggregateByKey` over `groupByKey` for aggregation.
- Filter and project before wide transformations.
- Avoid `collect()` on large RDDs; use `take()` or write output.
- Watch for skew in hot keys; salting may be needed.
- Persist reused RDDs and unpersist when done.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: groupByKey vs reduceByKey shuffle volume

**Steps**

1. Create an RDD with a few hot keys and many values.
2. Run `groupByKey().mapValues(len).collect()` and inspect shuffle write.
3. Run `reduceByKey(_ + _).collect()` and inspect shuffle write.
4. Compare the Stages tab metrics.

**What to observe**: `reduceByKey` combines locally and usually writes less shuffle data.

**Healthy result**: You can explain map-side combining and why it matters.

**Problematic result**: `groupByKey` can materialize huge value lists and cause OOM.

**Engineering takeaway**: Choose the aggregation primitive based on shuffle behavior, not just readability.

---

## Trade-offs and Alternatives

### Advantages

- Expressive functional API
- Explicit control over shuffles
- Good for unstructured data

### Limitations

- No optimizer
- Python serialization
- Easy to create accidental shuffles

### When to use it

- Text processing
- Custom parsing
- Legacy pipelines

### When not to use it

- Structured aggregations that DataFrames handle
- Large `groupByKey`
- Interactive BI

### Alternatives inside Spark

- DataFrame `groupBy`/`agg`
- Spark SQL

### Alternatives outside Spark

- MapReduce
- pandas
- Flink

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Count Customers per Segment
**Problem Statement**: Using PairRDD operations, count the total number of customers in each segment (Consumer, Corporate, SMB).  
**Input / Context**: customers_rdd

In [ ]:
# Exercise 1: Segment counts

### Exercise 2 (Intermediate): Distinct Countries Count
**Problem Statement**: Extract all country values, deduplicate using distinct(), and count how many unique countries exist.  
**Input / Context**: customers_rdd

In [ ]:
# Exercise 2: Unique countries

### Exercise 3 (Advanced): Custom aggregateByKey for Average Calculation
**Problem Statement**: Compute the average customer_id value per segment using aggregateByKey without collecting intermediate lists.  
**Input / Context**: customers_rdd

In [ ]:
# Exercise 3: aggregateByKey for average

---

## Production Debugging Exercise

### The groupByKey OOM

**Scenario**: A sessionization job fails with an executor OOM in the reduce stage.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Inspect shuffle read size and the largest task in the Stages tab.
2. Check whether one session has far more events than others.
3. Replace `groupByKey` with `aggregateByKey` or pre-aggregate.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Count Customers per Segment):
```python
segment_counts = customers_rdd.map(lambda c: (c[3], 1)).reduceByKey(lambda a, b: a + b).collect()
print("Segment breakdown:", segment_counts)
```

#### Solution 2 (Distinct Countries Count):
```python
unique_countries = customers_rdd.map(lambda c: c[1]).distinct()
print(f"Unique countries count: {unique_countries.count()}")
print("Sample countries:", unique_countries.take(5))
```

#### Solution 3 (Custom aggregateByKey for Average Calculation):
```python
# (sum, count) accumulator
zero_val = (0, 0)
seq_op = lambda acc, val: (acc[0] + val, acc[1] + 1)
comb_op = lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])

avg_per_segment = (
    customers_rdd
    .map(lambda c: (c[3], c[0]))
    .aggregateByKey(zero_val, seq_op, comb_op)
    .mapValues(lambda total_cnt: round(total_cnt[0] / total_cnt[1], 2))
    .collect()
)
print("Average ID per segment:", avg_per_segment)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Transformation vs Action**: Transformations define the recipe; Actions bake the cake.
- **Map-Side Combine**: reduceByKey combines rows locally on the mapper before network transmission.
- **Lazy Evaluation**: Allows Spark to pipeline consecutive map/filter steps into a single pass.

### Essential Technical Terms
- `PairRDD`: RDD containing key-value 2-tuples.
- `Map-Side Combine`: Local pre-aggregation performed in map task memory.
- `groupByKey`: Wide transformation that groups all values for each key without local aggregation.

### Common Engineering Mistakes
- **Using groupByKey for sums/counts**: Causes out-of-memory errors by transferring unaggregated records.
- **Calling collect() on big RDDs**: Crashes Driver JVM.

### Interview Questions on This Topic
- **Q: Why is reduceByKey better than groupByKey?**
  - *A*: reduceByKey performs partial aggregation on the map side before shuffling, minimizing network data transfer.
- **Q: What is flatMap used for?**
  - *A*: Splitting each element into 0, 1, or multiple elements (e.g. word tokenization).




### More Interview Questions

- **Q: What is the difference between `map` and `flatMap`?**
  - *A*: `map` returns one output per input; `flatMap` returns zero or more outputs per input and flattens them.
- **Q: Why is `reduceByKey` usually better than `groupByKey`?**
  - *A*: It combines values locally on the map side before the shuffle, reducing network traffic and memory pressure.

### What to Study Next
Open 02_rdd/03_rdd_partitions.ipynb to master partition mechanics and distribution.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')